# demo_liar

> Phase 15 (`specs/multiagent/phase15/`): why do larger Claude teams lose more to liars? Phase 14's Part B `baseline` cells are replayed from the response cache with a tracer (no API calls), the numpy forecast's cells are rerun with the same tracer, the honest loss is split into eight terms, and three rule-agent variants each copy one candidate mechanism (`TRACE_PLAN`).

Artifacts (gitignored): `results/multiagent/liar/{quick,full}/` (`trace_summary.json`, `accounting.csv`, `variants.json`, `report.md`, `liar.png`); the full run's `trace_summary.json` and `variants.json` are copied to `specs/multiagent/phase15/`. The replay reads Phase 10's response cache (`results/multiagent/llm/cache`) through the cache-only client: a miss raises, nothing is spent, and no key or `EXPLRL_LLM_LIVE` is needed.

- **Quick** (normal tests): every part with the fake client (Haiku cells: seeds 900-901, 4 turns) and short numpy grids.
- **Full** (`--cell-timeout 0 --flags llm`): the 30 Haiku cells (cache-only) with AC-4's checks against Phase 14's `runs.csv` and `turns.jsonl`, the 200 numpy cells against `scale_numpy.json`, the variants; only after `TRACE_PLAN` is frozen and committed.
- **Part D** (optional, `--flags llm` with `EXPLRL_PHASE15_D=1` and `EXPLRL_LLM_LIVE=1`): live, about $3.20, only after its forecast is committed and the user has approved.

In [ ]:
import json
import os
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

from explRL.bench_grid import readCSV, rowKey, writeCSV
from explRL.clue_runs import runClueAny
from explRL.coop_grid import coopHash
from explRL.liar_grid import TRACE_PLAN, forecastCells, liarForecast, liarShareGrid, nChange, numpyChanges, traceGrid, _pairs
from explRL.liar_report import MEASURES, TERMS, figTrace, lossAccount, mechanics, traceSummaries, writeTrace
from explRL.liar_trace import VARIANTS, traceCells
from explRL.llm_budget import Budget
from explRL.llm_sched import runWorkload
from explRL.train import demo_artifact_dir

LIAR = 'multiagent/liar'
WORKERS = min(8, os.cpu_count() or 1)
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
ROOT = LLM.parents[2]
SPEC = ROOT / 'specs/multiagent/phase15'
SCALE_FULL = demo_artifact_dir('full', 'multiagent/scale')      # Phase 14 Part B: runs.csv and turns.jsonl
NUMPY14 = ROOT / 'results/multiagent/forecast/scale_numpy.json'   # the numpy forecast's rows
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def committed(*paths):
    "Tracked by git and unchanged since the last commit."
    run = lambda *a: subprocess.run(['git', *a], cwd=ROOT, capture_output=True, text=True)
    return all(run('ls-files', str(p)).stdout.strip() and not run('status', '--porcelain', str(p)).stdout.strip() for p in paths)


def accounting(name, runs):
    "Per board (N, seed): the measures and the eight terms, one CSV row each."
    return [dict(runner=name, N=N, seed=sd, **mechanics(d['liar'].trace, d['liar'].board, d['liar'].nLiar),
                 **lossAccount(d['clean'], d['liar'])) for N, s in sorted(_pairs(runs).items()) for sd, d in sorted(s.items()) if len(d) == 2]


def analyse(mode, haiku, numpy, variants):
    "Summaries, report, figure and the per-board accounting in results/multiagent/liar/<mode> (AC-3's exact sum checked)."
    out = demo_artifact_dir(mode, LIAR)
    S = traceSummaries(haiku, numpy, variants)
    meta = dict(mode=mode, planHash=coopHash(TRACE_PLAN), commit=gitCommit(), haikuCells=len(haiku), numpyCells=len(numpy),
                haikuSource='fake client' if mode == 'quick' else 'replayed from the response cache')
    acc = accounting('haiku', haiku) + accounting('numpy', numpy)
    assert all(abs(r['total'] - r['loss']) < 1e-12 for r in acc), 'the eight terms must sum to the honest loss'
    writeTrace(out / 'report.md', S, meta)
    figTrace(out, S)
    writeCSV(acc, out / 'accounting.csv')
    (out / 'trace_summary.json').write_text(json.dumps(dict(meta, perN=S['perN'], change=S['change'], pooled=S['pooled'],
                                                            diff=S['diff'], hyp=S['hyp']),
                                                       indent=1, sort_keys=True, default=str) + '\n')
    (out / 'variants.json').write_text(json.dumps(S['variants'], indent=1, sort_keys=True, default=str) + '\n')
    return S, out


def short(cells, turns, seeds):
    return [dict(c, cfg=dict(c['cfg'], turns=turns)) for c in cells if c['seed'] in seeds]

## Quick: every part with the fake client

In [ ]:
t0 = time.time()
hq = traceCells(short(traceGrid(), 4, (900, 901)), workers=WORKERS)
nq = traceCells(forecastCells(sizes=(4, 8, 32), seeds=range(2000, 2004)), workers=WORKERS)
vq = {v: [tr.rows[0] for tr in traceCells(forecastCells(v, sizes=(8, 32), seeds=range(2000, 2004))).values()]
      for v in VARIANTS if v}
S, out = analyse('quick', hq, nq, vq)
assert time.time() - t0 < 180
for f in ('report.md', 'liar.png', 'accounting.csv', 'trace_summary.json', 'variants.json'):
    assert (out / f).stat().st_size > 0, f
assert set(S['hyp']) == {'H15_1', 'H15_2', 'H15_3', 'H15_4'} and set(S['variants']) == {'relay', 'first', 'repost'}
assert set(S['perN']['haiku']) == {8, 16, 32} and set(S['perN']['numpy']) == {4, 8, 32}
acc = readCSV(out / 'accounting.csv')
assert len(acc) == 3 * 2 + 3 * 4 and set(S['pooled']) == {'haiku', 'numpy'}
# identical with one process
ref = (out / 'trace_summary.json').read_text()
analyse('quick', traceCells(short(traceGrid(), 4, (900, 901))), traceCells(forecastCells(sizes=(4, 8, 32), seeds=range(2000, 2004))), vq)
assert (out / 'trace_summary.json').read_text().replace(gitCommit(), '') == ref.replace(gitCommit(), '')
print(open(out / 'report.md').read()[:1500])

## Full: replay and trace (`--cell-timeout 0 --flags llm`, $0)

`TRACE_PLAN` must be frozen and committed first. AC-4: every Haiku cell replays from cache; its row equals Phase 14's `runs.csv` row and its turn log equals the lines of Phase 14's `turns.jsonl`; NFR-1: the numpy rows equal `scale_numpy.json`.

In [ ]:
#| llm
def sameRows(traced, stored):
    "Each traced row's columns equal the stored row with the same key (both read back through CSV)."
    with tempfile.TemporaryDirectory() as d:
        writeCSV(traced, Path(d) / 'r.csv')
        mine = readCSV(Path(d) / 'r.csv')
    idx = {rowKey(r): r for r in stored}
    eq = lambda a, b: a == b or (a != a and b != b)
    return all(rowKey(r) in idx and all(eq(v, idx[rowKey(r)].get(k)) for k, v in r.items()) for r in mine)


def sameLogs(runs, path):
    "The stored turn-log lines of these keys, in order, equal the traced logs."
    want = {k: [json.dumps(x, sort_keys=True) + '\n' for x in tr.log] for k, tr in runs.items()}
    got = {k: [] for k in want}
    with open(path) as f:
        for line in f:
            k = json.loads(line)['key']
            if k in got:
                got[k].append(line)
    return got == want


if not (LLM / 'cache').exists() or not (SCALE_FULL / 'runs.csv').exists():
    print("skipped: needs Phase 14's cache and main run on this machine")
else:
    assert TRACE_PLAN['frozen'], 'freeze and commit TRACE_PLAN before the full traces'
    assert committed(ROOT / 'nbs/liar_grid.ipynb', ROOT / 'explRL/liar_grid.py'), 'commit the frozen TRACE_PLAN first'
    spent, t0 = Budget(CAP, LIVE_SPEC['ledger']).spent(), time.time()
    haiku = traceCells(traceGrid(), cache=LLM / 'cache', workers=WORKERS)
    assert len(haiku) == 30 and sameRows([tr.rows[0] for tr in haiku.values()], readCSV(SCALE_FULL / 'runs.csv'))
    assert sameLogs(haiku, SCALE_FULL / 'turns.jsonl'), 'traced turn logs differ from the stored ones'
    haiku = {k: tr._replace(log=[]) for k, tr in haiku.items()}
    numpy = traceCells(forecastCells(), workers=WORKERS)
    ref = {(r['N'], r['seed'], r['kind']): r for r in json.loads(NUMPY14.read_text())}
    for tr in numpy.values():
        r = tr.rows[0]
        x = ref[(int(r['N']), int(r['seed']), r['level'])]
        assert all(r[m] == x[m] or (r[m] != r[m] and x[m] != x[m]) for m in ('pooledShare', 'welfare', 'welfareHonest')), r['level']
    variants = {v: [tr.rows[0] for tr in traceCells(forecastCells(v), workers=WORKERS).values()] for v in VARIANTS if v}
    S, out = analyse('full', haiku, numpy, variants)
    assert Budget(CAP, LIVE_SPEC['ledger']).spent() == spent, 'the replay must spend nothing'
    for f in ('trace_summary.json', 'variants.json'):
        shutil.copy(out / f, SPEC / f)
    print(f'full: {time.time() - t0:.0f} s, AC-4 and NFR-1 hold, nothing spent')
    print(open(out / 'report.md').read()[:6000])

## Part D (optional; `--flags llm`, `EXPLRL_PHASE15_D=1`, `EXPLRL_LLM_LIVE=1`; about 7 000 calls, about $3.20)

FR-D2: the variant is the one, among those passing H15-4, whose honest-loss change is closest to Haiku's; none passes: no Part D. The first pass writes `liar_predictions.json` (numpy, seconds) and stops; commit it, then run again for the live cells and the score.

In [ ]:
#| llm
PRED = SPEC / 'liar_predictions.json'
full = demo_artifact_dir('full', LIAR)
if os.environ.get('EXPLRL_PHASE15_D') != '1' or not (full / 'trace_summary.json').exists():
    print('skipped: Part D needs EXPLRL_PHASE15_D=1, the full traces and the user\'s approval')
else:
    T, V = json.loads((full / 'trace_summary.json').read_text()), json.loads((full / 'variants.json').read_text())
    obs = T['change']['haiku']['loss']
    ok = {v: x for v, x in V.items() if x['honestLoss']['change']['sign'] == obs['sign']}
    if not ok:
        print('no variant passes H15-4: Part D is not run')
    elif not PRED.exists():
        best = min(ok, key=lambda v: abs(ok[v]['honestLoss']['change']['mean'] - obs['mean']))
        PRED.write_text(json.dumps(dict(variant=best, forecast=liarForecast(best, workers=WORKERS)), indent=1, sort_keys=True) + '\n')
        print(f'forecast written for variant {best}: commit {PRED.name}, then run this section again')
    else:
        assert committed(PRED), f'commit {PRED.name} before the live cells'
        assert os.environ.get('EXPLRL_LLM_LIVE') == '1', 'Part D makes live calls: set EXPLRL_LLM_LIVE=1'
        out = demo_artifact_dir('full', 'multiagent/liarD')
        res = runWorkload([('liarD', liarShareGrid())], LIVE_SPEC, 32, str(out / 'turns'), run=runClueAny)
        writeCSV(res['rows'], out / 'runs.csv')
        spent, ref = Budget(CAP, LIVE_SPEC['ledger']).spent(), (out / 'runs.csv').read_bytes()
        again = runWorkload([('liarD', liarShareGrid())], LIVE_SPEC, 32, str(out / 'turns'), run=runClueAny)
        writeCSV(again['rows'], out / 'runs.csv')               # the cached rerun: byte-identical and free
        assert (out / 'runs.csv').read_bytes() == ref and Budget(CAP, LIVE_SPEC['ledger']).spent() == spent
        clean = [r for r in readCSV(SCALE_FULL / 'runs.csv')
                 if r['study'] == 'scaleB' and r['level'] == 'baseline' and int(r['N']) in (8, 32)]
        got = numpyChanges(res['rows'] + clean, ('honestLoss',))['honestLoss']['change']
        pred = json.loads(PRED.read_text())
        score = dict(observed=got, predicted={k: v['change'] for k, v in pred['forecast'].items()},
                     agree={k: v['change']['sign'] == got['sign'] for k, v in pred['forecast'].items()})
        (out / 'score.json').write_text(json.dumps(score, indent=1, sort_keys=True) + '\n')
        print(json.dumps(score, indent=1))